# pulsar — 使用教程

**pulsar** 是一个运行在终端中的实时硬件监控仪表盘，专为需要在终端里监控计算资源的开发者和研究人员设计。

它能显示：
- 每个 CPU 核心的负载（带颜色进度条）
- 内存使用情况
- 磁盘 I/O 吞吐量（实时 MB/s）
- 占用资源最多的进程列表

本 notebook 演示如何通过 Python API 和命令行两种方式使用 pulsar。

## 1. 安装

In [ ]:
# 在项目根目录下以可编辑模式安装
# 只需执行一次
import subprocess, sys
result = subprocess.run(
    [sys.executable, "-m", "pip", "install", "-e", "."],
    capture_output=True, text=True
)
print(result.stdout[-500:] if len(result.stdout) > 500 else result.stdout)
if result.returncode != 0:
    print("STDERR:", result.stderr[-300:])

## 2. 项目结构一览

```
pulsar/
├── pyproject.toml          # 包配置 & 依赖声明
├── src/pulsar/
│   ├── __init__.py         # 公开 API 入口
│   ├── cli.py              # 命令行接口 (click)
│   ├── collector.py        # 硬件数据采集 (psutil)
│   ├── dashboard.py        # 实时 TUI 仪表盘 (rich.live)
│   └── snapshot.py         # 单次快照输出 (table / json)
└── tests/
```

数据流：
```
cli.py
  ├── --once  →  collector.collect()  →  snapshot.print_table() / print_json()
  └── (live)  →  dashboard.run()  →  循环调用 collector.collect()
```

## 3. Python API — 直接调用采集函数

### 3.1 读取静态硬件信息（启动时调用一次）

In [ ]:
from pulsar import get_system_info

info = get_system_info()
print(f"CPU 型号   : {info.cpu_model}")
print(f"逻辑核心数  : {info.cpu_cores}")
print(f"当前频率    : {info.cpu_freq_current} GHz  (最大 {info.cpu_freq_max} GHz)")
print(f"GPU        : {info.gpu_model}")
print(f"总内存      : {info.ram_total} GB")

### 3.2 采集一次实时快照

In [ ]:
import time
from pulsar import collect

# 第一次调用用于初始化磁盘 I/O 基准值（返回 0.0 MB/s 属正常）
collect()
time.sleep(0.5)

# 第二次调用才能得到有意义的磁盘 I/O delta
snap = collect(top_n=5)

print("=== CPU ===")
for i, pct in enumerate(snap.cpu_per_core):
    bar = '█' * int(pct / 5) + '░' * (20 - int(pct / 5))
    print(f"  Core {i:>2}  {bar}  {pct:5.1f}%")
print(f"  平均     {snap.cpu_avg:.1f}%")

print("\n=== 内存 ===")
print(f"  已用: {snap.mem_used} GB / 总计: {snap.mem_total} GB  ({snap.mem_percent:.1f}%)")

print("\n=== 磁盘 I/O ===")
print(f"  读取: {snap.disk_read_mbps:.2f} MB/s")
print(f"  写入: {snap.disk_write_mbps:.2f} MB/s")

print("\n=== Top 5 进程 ===")
print(f"  {'PID':>7}  {'Name':<24}  {'CPU%':>6}  {'MEM':>10}")
for p in snap.top_procs:
    mem_str = f"{p['mem_mb']} MB" if p['mem_mb'] < 1024 else f"{p['mem_mb']/1024:.2f} GB"
    print(f"  {p['pid']:>7}  {p['name']:<24}  {p['cpu']:>5.1f}%  {mem_str:>10}")

### 3.3 Snapshot 数据结构说明

`collect()` 返回一个 `Snapshot` dataclass，字段含义如下：

| 字段 | 类型 | 说明 |
|---|---|---|
| `cpu_per_core` | `list[float]` | 每个逻辑核心的使用率（%） |
| `cpu_avg` | `float` | 所有核心的平均使用率（%） |
| `mem_used` | `float` | 已用内存（GB） |
| `mem_total` | `float` | 总内存（GB） |
| `mem_percent` | `float` | 内存使用率（%） |
| `disk_read_mbps` | `float` | 磁盘读取速率（MB/s） |
| `disk_write_mbps` | `float` | 磁盘写入速率（MB/s） |
| `top_procs` | `list[dict]` | 进程列表，每项含 `pid`, `name`, `cpu`, `mem_mb` |
| `timestamp` | `float` | Unix 时间戳 |

### 3.4 按名称过滤进程

In [ ]:
# 只查看名称含 "python" 的进程
snap_py = collect(top_n=10, proc_filter=["python"])

if snap_py.top_procs:
    print(f"找到 {len(snap_py.top_procs)} 个 Python 进程：")
    for p in snap_py.top_procs:
        print(f"  PID {p['pid']:>6}  {p['name']:<20}  CPU {p['cpu']:.1f}%  MEM {p['mem_mb']:.0f} MB")
else:
    print("当前没有匹配 'python' 的进程在运行")

### 3.5 连续采样 — 监控 CPU 10 秒

In [ ]:
import time
from pulsar import collect

# 预热
collect()
time.sleep(0.3)

samples = []
n_samples = 10

print(f"采集 {n_samples} 次 CPU 数据（每隔 1 秒）...")
for i in range(n_samples):
    snap = collect()
    samples.append(snap.cpu_avg)
    print(f"  [{i+1:>2}/{n_samples}]  avg CPU: {snap.cpu_avg:5.1f}%  "
          f"mem: {snap.mem_percent:.1f}%  "
          f"disk R: {snap.disk_read_mbps:.2f} MB/s")
    time.sleep(1.0)

print(f"\n10 秒统计：")
print(f"  最小 CPU: {min(samples):.1f}%")
print(f"  最大 CPU: {max(samples):.1f}%")
print(f"  平均 CPU: {sum(samples)/len(samples):.1f}%")

## 4. 命令行用法

安装后即可在终端使用 `pulsar` 命令。以下展示各种用法。

### 4.1 查看帮助

In [ ]:
!pulsar --help

### 4.2 `--once` 模式：单次快照（table 格式）

适用于脚本调用或快速查看当前状态，输出后立即退出。

In [ ]:
!pulsar --once

### 4.3 `--once --format json`：机器可读的 JSON 快照

适合日志记录、数据管道或与其他工具集成。

In [ ]:
!pulsar --once --format json

### 4.4 将 JSON 快照导入 Python 分析

In [ ]:
import subprocess, json

result = subprocess.run(
    ["pulsar", "--once", "--format", "json"],
    capture_output=True, text=True
)
data = json.loads(result.stdout)

print("JSON 键结构：", list(data.keys()))
print(f"\n时间戳    : {data['timestamp']:.2f}")
print(f"CPU 型号  : {data['system']['cpu_model']}")
print(f"平均 CPU  : {data['cpu']['avg_percent']}%")
print(f"内存使用率 : {data['memory']['percent']}%")
print(f"磁盘读取  : {data['disk_io']['read_mbps']} MB/s")
print(f"Top 进程数 : {len(data['top_processes'])}")

### 4.5 `--top N`：控制显示的进程数量

In [ ]:
# 显示 top 10 进程
!pulsar --once --top 10

### 4.6 `--proc NAME`：过滤进程（可重复使用）

In [ ]:
# 只显示名称含 "python" 或 "kernel" 的进程
!pulsar --once --proc python --proc kernel

### 4.7 `--interval`：调整刷新频率

| 场景 | 推荐 interval |
|---|---|
| 常规监控 | `1.0`（默认）|
| 高频采样 | `0.5` 或 `0.25` |
| 低功耗模式 | `2.0` 或 `5.0` |

```bash
pulsar --interval 0.5   # 每 0.5 秒刷新一次
```

**注意：** `--interval` 在 `--once` 模式下也会影响磁盘 I/O 预热的等待时间（最多 0.5 秒）。

### 4.8 实时仪表盘模式

直接运行 `pulsar` 即可启动实时 TUI 仪表盘（需要真实终端，**不能在 notebook 内运行**）：

```bash
# 基础启动
pulsar

# 快速刷新 + 只看 python 进程
pulsar --interval 0.5 --proc python

# 显示 top 10 进程
pulsar --top 10

# 彩虹 disco 模式
pulsar --disco
```

仪表盘快捷键：

| 按键 | 功能 |
|---|---|
| `y` | 触发烟花动画 |
| `q` 或 `Ctrl-C` | 退出 |

## 5. 进阶：定时记录 CPU/内存数据

In [ ]:
import time, json
from datetime import datetime
from pulsar import collect

# 采集 5 次，每次间隔 1 秒，保存为 JSON Lines 格式
collect()          # 预热磁盘 I/O
time.sleep(0.3)

log_path = "/tmp/pulsar_log.jsonl"
with open(log_path, "w") as f:
    for _ in range(5):
        snap = collect(top_n=3)
        record = {
            "time": datetime.now().isoformat(),
            "cpu_avg": snap.cpu_avg,
            "mem_percent": snap.mem_percent,
            "disk_read_mbps": snap.disk_read_mbps,
            "disk_write_mbps": snap.disk_write_mbps,
        }
        f.write(json.dumps(record) + "\n")
        print(record)
        time.sleep(1.0)

print(f"\n已保存到 {log_path}")

## 6. 进阶：用 matplotlib 可视化采集数据

In [ ]:
import time
import matplotlib.pyplot as plt
from pulsar import collect

collect()
time.sleep(0.3)

timestamps = []
cpu_data   = []
mem_data   = []

print("采集 15 秒数据...")
for i in range(15):
    snap = collect()
    timestamps.append(i)
    cpu_data.append(snap.cpu_avg)
    mem_data.append(snap.mem_percent)
    time.sleep(1.0)

fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(10, 5), sharex=True)

ax1.plot(timestamps, cpu_data, color="steelblue", linewidth=2, marker="o", markersize=4)
ax1.set_ylabel("CPU 使用率 (%)")
ax1.set_ylim(0, 100)
ax1.axhline(80, color="orange", linestyle="--", linewidth=0.8, label="80% 警戒线")
ax1.legend(fontsize=9)
ax1.grid(True, alpha=0.3)
ax1.set_title("pulsar — 15 秒硬件监控")

ax2.plot(timestamps, mem_data, color="tomato", linewidth=2, marker="s", markersize=4)
ax2.set_ylabel("内存使用率 (%)")
ax2.set_xlabel("时间 (秒)")
ax2.set_ylim(0, 100)
ax2.axhline(90, color="orange", linestyle="--", linewidth=0.8, label="90% 警戒线")
ax2.legend(fontsize=9)
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig("/tmp/pulsar_plot.png", dpi=150)
plt.show()
print("图表已保存到 /tmp/pulsar_plot.png")

## 7. 各选项速查表

| 命令 | 说明 |
|---|---|
| `pulsar` | 启动实时 TUI 仪表盘 |
| `pulsar --once` | 单次快照（table 格式）后退出 |
| `pulsar --once --format json` | 单次快照（JSON 格式）后退出 |
| `pulsar --once --format json > snap.json` | 快照保存为文件 |
| `pulsar --interval 0.5` | 每 0.5 秒刷新一次 |
| `pulsar --top 10` | 显示 Top 10 进程 |
| `pulsar --proc python` | 只显示名称含 `python` 的进程 |
| `pulsar --proc python --proc node` | 同时过滤多个进程名 |
| `pulsar --disco` | 彩虹 Disco 模式 |
| `pulsar --help` | 查看帮助 |

**仪表盘快捷键：**
- `y` — 烟花动画（或 CPU 核心持续满载 5 秒时自动触发）
- `q` / `Ctrl-C` — 退出